# S2.6 — Shuffle Deep Dive
**Date completed:** September 2026  
**Status:** In Progress  
**Interview covered:** Q13 — What is shuffling? What operations cause it?

In [0]:
# ============================================================
# Cell 2 — Causing a Shuffle deliberately
# Goal: See shuffle in Query Profile and measure its cost
# Operation: groupBy = WIDE transformation = causes shuffle
# ============================================================

import time
import pyspark.sql.functions as F

print("=== SHUFFLE DEEP DIVE ===")
print()

# Step 1 — Create 10M row dataset with 4 cities
df = spark.range(0, 10000000)

df = df.withColumn("city",
    F.when(df.id % 4 == 0, "Delhi")
    .when(df.id % 4 == 1, "Mumbai")
    .when(df.id % 4 == 2, "Pune")
    .otherwise("Chennai"))

df = df.withColumn("order_value",
    (df.id % 1000).cast("double"))

print("Dataset created — 10M rows, 4 cities")
print(f"Default shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print()

# Step 2 — Trigger shuffle with groupBy (WIDE transformation)
# groupBy needs ALL same-city rows on ONE executor
# This forces data movement = shuffle
print("=== Triggering SHUFFLE with groupBy ===")
start = time.time()

result = df.groupBy("city").agg(
    F.count("*").alias("total_orders"),
    F.sum("order_value").alias("total_revenue"),
    F.avg("order_value").alias("avg_order_value")
)
result.show()

end = time.time()
print(f"Time taken: {round(end - start, 4)} seconds")
print("Check Query Profile — observe Shuffle step")

In [0]:
# ============================================================
# Cell 3 — Optimisation: Filter BEFORE groupBy
# Goal: Prove that filtering early reduces groupBy cost
# Rule: Reduce rows BEFORE expensive wide transformation
# ============================================================

print("=== OPTIMISATION — Filter BEFORE groupBy ===")
print()

# Approach 1 — No filter — all 10M rows go into groupBy
print("--- Approach 1: No filter (all 10M rows grouped) ---")
start = time.time()

df.groupBy("city") \
  .agg(F.count("*").alias("total_orders")) \
  .show()

end = time.time()
time_without_filter = round(end - start, 4)
print(f"Time taken: {time_without_filter} seconds")
print()

# Approach 2 — Filter first — only 2.5M Delhi rows go into groupBy
print("--- Approach 2: Filter first (2.5M rows only) ---")
start = time.time()

df.filter(F.col("city") == "Delhi") \
  .groupBy("city") \
  .agg(F.count("*").alias("total_orders")) \
  .show()

end = time.time()
time_with_filter = round(end - start, 4)
print(f"Time taken: {time_with_filter} seconds")
print()

# Summary
print("=== COMPARISON ===")
print(f"Without filter : {time_without_filter} seconds (10M rows grouped)")
print(f"With filter    : {time_with_filter} seconds (2.5M rows grouped)")
print(f"Improvement    : {round(time_without_filter - time_with_filter, 4)} seconds saved")

## Key Takeaways — S2.6 Shuffle Deep Dive

## What is Shuffle
Data physically moving between executors across the network.
Required when an operation needs rows from MULTIPLE partitions.

## Library Analogy
Books (rows) travelling between floors (executors) to be grouped by subject (key).

## Operations that cause Shuffle
| Operation | Why shuffle needed |
|-----------|-------------------|
| groupBy() | All same-key rows must meet on one executor |
| join() | Matching rows from both DataFrames must meet |
| distinct() | All rows needed to find duplicates |
| orderBy() | All rows needed for global sort |

## Operations that do NOT cause Shuffle
filter(), select(), withColumn(), when/otherwise, map()

## Query Profile Findings
- Input: 10M rows
- Partial Aggregate: 32 rows (8 partitions × 4 cities)
- Shuffle: 32 rows, 4ms — TINY
- Grouping Aggregate: 199ms — MOST EXPENSIVE
- Shuffle is NOT always the bottleneck — grouping cost is

## Optimisation Rule
Filter BEFORE groupBy — reduce rows before expensive operation
In-memory: small saving | On disk (Delta/Parquet): massive saving

## Broadcast Join Preview (S4.3)
Small table (500 rows) → broadcast to all executors → no shuffle on big table
Large table (10M rows) → stays in place → never moved